In [23]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers import CommaSeparatedListOutputParser

load_dotenv()
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

In [24]:
# JSON Parser
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

output_parser = JsonOutputParser(pydantic_object=Joke)
format_instructions = output_parser.get_format_instructions()


In [25]:
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | llm | output_parser
result = chain.invoke({"query": "Tell me a joke"})
print("JSON result:", result)
print("Setup:", result["setup"])
print("Punchline:", result["punchline"])

print("---")

JSON result: {'setup': "Why couldn't the bicycle stand up by itself?", 'punchline': 'Because it was two-tired.'}
Setup: Why couldn't the bicycle stand up by itself?
Punchline: Because it was two-tired.
---


In [26]:
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"setup": {"title": "Setup", "description": "question to set up a joke", "type": "string"}, "punchline": {"title": "Punchline", "description": "answer to resolve the joke", "type": "string"}}, "required": ["setup", "punchline"]}
```


In [27]:
# CSV Parser
csv_parser = CommaSeparatedListOutputParser()
format_instructions = csv_parser.get_format_instructions()

prompt2 = PromptTemplate(
    template="Answer the user query. {format_instructions}\nList five {subject}.",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

chain2 = prompt2 | llm | csv_parser
result2 = chain2.invoke({"subject": "ice cream flavors"})
print("CSV result:", result2)
print("First flavor:", result2[0])

CSV result: ['Vanilla', 'Chocolate', 'Strawberry', 'Cookies and Cream', 'Mint Chocolate Chip']
First flavor: Vanilla


In [29]:
print(format_instructions)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


Question. Where is the formal instructions coming from? Like I didn't give any instructions..<br>
Answer:<br>
This line: pythonformat_instructions = output_parser.get_format_instructions()<br>
The output parser generates the instructions automatically. <br>
We never wrote them, the parser knows what format it needs and writes the instructions itself.<br>
Those instructions get injected into the prompt via partial_variables. <br>
So the LLM receives them and knows exactly how to format its response.<br>
Basically, <br>
we defined the shape (the Joke class), the parser figured out the instructions automatically.<br>
